<a href="https://colab.research.google.com/github/OlajideFemi/Carbon-Footprint/blob/main/Mojibake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [ ]:
df.to_csv("salesforce_clean.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
import unicodedata

def clean_text(value):
    if isinstance(value, str):
        value = unicodedata.normalize("NFKD", value)
        value = value.replace("Â", "")
        value = value.replace("Ã", "")
        value = value.replace("â€™", "'")
        value = value.replace("â€“", "-")
        value = value.strip()
    return value

df = df.applymap(clean_text)

In [ ]:
import re

def remove_weird_chars(text):
    if isinstance(text, str):
        return re.sub(r"[^\x20-\x7E]+", "", text)
    return text

df = df.applymap(remove_weird_chars)

In [ ]:
from simple_salesforce import Salesforce
import pandas as pd

data = sf.query_all("SELECT Id, Name FROM Account")

df = pd.DataFrame(data["records"]).drop(columns=["attributes"])

df.to_csv("salesforce_data.csv", index=False, encoding="utf-8-sig")

BrokenActualâ€™’â€“–Ã©éÂ££

In [ ]:
df = df.applymap(lambda x: x.encode('latin1').decode('utf8') if isinstance(x,str) else x)

In [ ]:
def clean_salesforce_dataframe(df):

    import unicodedata
    import re

    def clean(value):
        if isinstance(value, str):
            value = unicodedata.normalize("NFKC", value)
            value = re.sub(r"[^\x20-\x7E£€]", "", value)
            value = value.strip()
        return value

    return df.applymap(clean)

df = clean_salesforce_dataframe(df)

In [ ]:
import pandas as pd
import re
import unicodedata


def fix_mojibake(text):
    """
    Try to repair text that was decoded with the wrong encoding.
    Example: 'â€™' -> '’', 'Ã©' -> 'é', 'Â£' -> '£'
    """
    if not isinstance(text, str):
        return text

    original = text

    # Only attempt repair if common mojibake markers appear
    suspicious_markers = ["Ã", "Â", "â", "ð", "�"]
    if any(marker in text for marker in suspicious_markers):
        for src_enc, dst_enc in [("latin1", "utf-8"), ("cp1252", "utf-8")]:
            try:
                repaired = text.encode(src_enc, errors="ignore").decode(dst_enc, errors="ignore")
                if repaired and repaired != text:
                    text = repaired
                    break
            except Exception:
                pass

    return text if text else original


def clean_unicode_text(text):
    """
    General text cleanup after mojibake repair.
    """
    if not isinstance(text, str):
        return text

    # Repair bad decoding first
    text = fix_mojibake(text)

    # Normalize unicode
    text = unicodedata.normalize("NFKC", text)

    # Replace non-breaking spaces with normal spaces
    text = text.replace("\u00A0", " ")

    # Remove zero-width / invisible characters
    text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)

    # Remove ASCII control chars except tab/newline if you want to keep them
    text = re.sub(r"[\x00-\x1F\x7F]", "", text)

    # Strip outer whitespace
    text = text.strip()

    return text


def clean_salesforce_dataframe(df):
    """
    Clean all object/string columns in a DataFrame.
    """
    df = df.copy()

    object_cols = df.select_dtypes(include=["object", "string"]).columns

    for col in object_cols:
        df[col] = df[col].map(clean_unicode_text)

    return df


# Example usage
# df = pd.DataFrame(sf.query_all(query)["records"]).drop(columns=["attributes"], errors="ignore")
# df = clean_salesforce_dataframe(df)
# df.to_csv("salesforce_clean.csv", index=False, encoding="utf-8-sig")

In [ ]:
def clean_salesforce_dataframe_with_report(df, sample_rows=5):
    cleaned_df = clean_salesforce_dataframe(df)

    object_cols = df.select_dtypes(include=["object", "string"]).columns
    changes = []

    for col in object_cols:
        before = df[col]
        after = cleaned_df[col]

        changed_mask = before.fillna("") != after.fillna("")
        changed_rows = df.loc[changed_mask, [col]].head(sample_rows)

        if not changed_rows.empty:
            for idx in changed_rows.index:
                changes.append({
                    "row_index": idx,
                    "column": col,
                    "before": before.loc[idx],
                    "after": after.loc[idx],
                })

    report_df = pd.DataFrame(changes)
    return cleaned_df, report_df


# Example:
# cleaned_df, report = clean_salesforce_dataframe_with_report(df)
# print(report.head(20))
# cleaned_df.to_csv("salesforce_clean.csv", index=False, encoding="utf-8-sig")

In [ ]:
from simple_salesforce import Salesforce
import pandas as pd

# Example query
result = sf.query_all("""
    SELECT Id, Name, BillingCity, Description
    FROM Account
""")

df = pd.DataFrame(result["records"]).drop(columns=["attributes"], errors="ignore")

# Clean text problems
df = clean_salesforce_dataframe(df)

# Export for Excel
df.to_csv("salesforce_clean.csv", index=False, encoding="utf-8-sig")

In [ ]:
print(df.head().to_dict())